# Task 3 — Thực nghiệm bộ giải Kakuro bằng CSP

Notebook này bổ sung bằng chứng thực nghiệm cho **bảng Kakuro 9×9 nâng cao trong đề**. Nội dung gồm mô hình CSP, kiểm tra nghiệm bằng các ràng buộc gốc, so sánh GAC với Forward Checking và kiểm tra tính ổn định qua nhiều lần chạy. Thời gian phụ thuộc phần cứng; không so sánh trực tiếp giữa các máy.

## Đối chiếu yêu cầu
- **Yêu cầu 1:** nêu Variables, Domains, Constraints và các kỹ thuật tìm kiếm/lan truyền được cài đặt.
- **Yêu cầu 2:** giải bảng 9×9 bằng hai cấu hình, xác nhận nghiệm với `validate_solution()` và ghi thời gian, nodes, assignments, backtracks, domain reductions, contradictions.
- **Yêu cầu 3:** giao diện Pygame hiện có ở `main.py` → **AI Solver**; giao diện trực quan miền/pencil marks, propagation, gán giá trị và quay lui qua sự kiện của solver.

Chạy notebook từ thư mục `task3/` hoặc thư mục gốc của repository. Không cần cài thêm thư viện ngoài `requirements.txt`.

## 1. Mô hình hóa thành CSP

Một CSP là bộ ba **(X, D, C)**:

- **Variables (X):** mỗi ô trắng là một biến `X_(r,c)` theo tọa độ bắt đầu từ 0. Một ô thuộc đồng thời một run ngang và một run dọc; hai run giao nhau tại cùng biến.
- **Domains (D):** ban đầu `D_(r,c) = {1,2,...,9}` (ô cho sẵn, nếu có, sẽ có miền singleton).
- **Constraints (C):** với mỗi run `R=(X1,...,Xk)` và clue `T`, `X1 + ... + Xk = T` đồng thời `AllDifferent(X1,...,Xk)`. Đây là ràng buộc n-ary; cả tổng và không lặp phải đúng trong cùng run.

Trong implementation, mỗi run tiền sinh các tuple chữ số hợp lệ (tổng đúng, không lặp). Một phép gán từng phần còn hợp lệ khi mỗi run vẫn có ít nhất một tuple hỗ trợ. Constraint graph có biến là đỉnh; các ô cùng run là láng giềng.

In [1]:
from pathlib import Path
import sys

from time import perf_counter

project_dir = Path.cwd()
if not (project_dir / "boards.py").is_file():
    project_dir = project_dir / "task3"
if not (project_dir / "boards.py").is_file():
    raise FileNotFoundError("Không tìm thấy task3/boards.py; hãy mở notebook trong repository.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from boards import make_assignment_board, make_demo_board
from kakuro_solver import KakuroSolver

board = make_assignment_board()
model = KakuroSolver(board).csp
print(board.describe())
print(f"Số biến (ô trắng): {len(model.variables)}")
print(f"Số ràng buộc run: {len(model.constraints)}")
print(f"Miền ban đầu mỗi ô: {sorted(next(iter(model.initial_domains().values())))}")
print(f"Tổng số cạnh constraint graph: {sum(map(len, model.constraint_graph.values())) // 2}")

Assignment Kakuro 9x9: 48 variables, 28 run constraints.
Số biến (ô trắng): 48
Số ràng buộc run: 28
Miền ban đầu mỗi ô: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Tổng số cạnh constraint graph: 166


## 2. Thuật toán và heuristic

- **GAC:** với từng run n-ary, bỏ một giá trị khỏi miền nếu không có tuple hợp lệ nào của run hỗ trợ giá trị đó; khi miền đổi, đưa các run giao với ô vào hàng đợi để kiểm tra lại.
- **Forward Checking:** sau khi gán một ô, cập nhật hỗ trợ trong các run chứa ô đó; phát hiện miền rỗng/mâu thuẫn sớm.
- **Backtracking Search:** gán biến chưa gán, lan truyền, đệ quy; nếu mâu thuẫn thì khôi phục bằng cách dùng bản sao miền và thử nhánh khác.
- **MRV:** chọn biến có miền nhỏ nhất; **Degree** phá hòa bằng số láng giềng chưa gán lớn nhất; **LCV** thử trước giá trị loại ít lựa chọn của láng giềng nhất.

Benchmark giữ nguyên MRV, Degree, LCV và chỉ đổi phương pháp propagation để cô lập ảnh hưởng của GAC so với Forward Checking.

In [2]:
CONFIGURATIONS = [
    ("GAC + MRV + Degree + LCV", "gac"),
    ("Forward Checking + MRV + Degree + LCV", "forward_checking"),
]
REPEATS = 2
benchmark_results = {}

for label, propagation in CONFIGURATIONS:
    trials = []
    for trial in range(1, REPEATS + 1):
        solver = KakuroSolver(
            board,
            propagation=propagation,
            use_mrv=True,
            use_degree=True,
            use_lcv=True,
        )
        started = perf_counter()
        solution = solver.solve()
        wall_ms = (perf_counter() - started) * 1000
        valid = solution is not None and solver.csp.validate_solution(solution)
        assert valid, f"{label}, lượt {trial}: không tìm được nghiệm hợp lệ"
        stats = solver.stats.as_dict()
        trials.append({"wall_ms": wall_ms, "stats": stats})
        print(
            f"{label} | lượt {trial}/{REPEATS} | hợp lệ={valid} | "
            f"{wall_ms:.3f} ms | nodes={stats['expanded_nodes']} | "
            f"assignments={stats['assignments_tried']} | "
            f"backtracks={stats['backtrack_count']}"
        )
    benchmark_results[label] = trials

print("\nTóm tắt (thời gian wall-clock; trung bình ± độ lệch chuẩn mẫu):")
print(f"{'Cấu hình':42} {'Valid':>5} {'ms mean ± sd':>20} {'Nodes':>8} {'Assign':>8} {'Backtrack':>10} {'Removed':>9} {'Contrad.':>9}")
for label, trials in benchmark_results.items():
    times = [trial["wall_ms"] for trial in trials]
    mean_ms = sum(times) / len(times)
    sd_ms = (sum((value - mean_ms) ** 2 for value in times) / (len(times) - 1)) ** 0.5 if len(times) > 1 else 0.0
    mean_stat = lambda key: sum(trial["stats"][key] for trial in trials) / len(trials)
    print(
        f"{label:42} {'Yes':>5} {mean_ms:9.3f} ± {sd_ms:<8.3f} "
        f"{mean_stat('expanded_nodes'):8.1f} {mean_stat('assignments_tried'):8.1f} "
        f"{mean_stat('backtrack_count'):10.1f} {mean_stat('domain_values_removed'):9.1f} "
        f"{mean_stat('contradictions'):9.1f}"
    )

assert all(len(trials) == REPEATS for trials in benchmark_results.values())
print(f"\nKết luận kiểm tra: cả {len(CONFIGURATIONS) * REPEATS} lần chạy đều trả nghiệm 9×9 hợp lệ.")

GAC + MRV + Degree + LCV | lượt 1/2 | hợp lệ=True | 25013.445 ms | nodes=49 | assignments=48 | backtracks=0
GAC + MRV + Degree + LCV | lượt 2/2 | hợp lệ=True | 24563.034 ms | nodes=49 | assignments=48 | backtracks=0
Forward Checking + MRV + Degree + LCV | lượt 1/2 | hợp lệ=True | 129211.783 ms | nodes=1218 | assignments=1635 | backtracks=1587
Forward Checking + MRV + Degree + LCV | lượt 2/2 | hợp lệ=True | 110149.721 ms | nodes=1218 | assignments=1635 | backtracks=1587

Tóm tắt (thời gian wall-clock; trung bình ± độ lệch chuẩn mẫu):
Cấu hình                                   Valid         ms mean ± sd    Nodes   Assign  Backtrack   Removed  Contrad.
GAC + MRV + Degree + LCV                     Yes 24788.240 ± 318.489      49.0     48.0        0.0     384.0       0.0
Forward Checking + MRV + Degree + LCV        Yes 119680.752 ± 13478.913   1218.0   1635.0     1587.0    9370.0     418.0

Kết luận kiểm tra: cả 4 lần chạy đều trả nghiệm 9×9 hợp lệ.


## 3. Kiểm tra sự kiện và giao diện trực quan

GUI hiện có nhận callback `SolverEvent`: `domain_reduced` cập nhật pencil marks/miền, `assignment` hiển thị quyết định gán, `contradiction` và `backtrack` cho thấy nhánh thất bại/quay lui, `solution_found` báo hoàn tất. Chạy `python main.py` trong `task3/`, chọn **AI Solver**, rồi dùng **Start**, **Pause/Resume**, **Step** và **Reset** để quan sát quá trình. Chọn GAC/Forward Checking/None và bật/tắt MRV, Degree, LCV để minh họa từng kỹ thuật.

Cell kế tiếp xác nhận solver phát ra các sự kiện miền, gán và nghiệm trên bảng demo nhỏ; việc xem hoạt động trực quan đầy đủ thực hiện trong GUI.

In [3]:
events = []
demo_solver = KakuroSolver(
    make_demo_board(),
    propagation="gac",
    event_callback=events.append,
)
demo_solution = demo_solver.solve()
assert demo_solution is not None and demo_solver.csp.validate_solution(demo_solution)
event_counts = {}
for event in events:
    event_counts[event.kind] = event_counts.get(event.kind, 0) + 1
print("Nghiệm bảng demo hợp lệ:", demo_solver.csp.validate_solution(demo_solution))
print("Sự kiện do solver phát ra:", event_counts)
assert "assignment" in event_counts and "solution_found" in event_counts

Nghiệm bảng demo hợp lệ: True
Sự kiện do solver phát ra: {'domain_reduced': 13, 'variable_selected': 4, 'assignment': 4, 'solution_found': 1}
